# 🛡️ Tank duel — PPO + GRU self-play on Colab
Train here (GPU for the network, all vCPUs for the arenas), duel on your laptop with `referee.py`.
Runtime → *Change runtime type* → **A100 GPU**. Run the cells top to bottom; after a disconnect run them again — training **resumes** from Google Drive.

In [ ]:
import os, multiprocessing
print("vCPU:", multiprocessing.cpu_count())
!nvidia-smi -L

## 1. Drive (checkpoints survive disconnects) and code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')   # only for checkpoints/exports (they survive disconnects)

RUN_DIR = '/content/drive/MyDrive/tank_runs/run1'   # change the name for a new experiment
ARENA_URL = 'https://github.com/BohatyrovIvanUcu/tank_battle_playground.git'   # shared arena + referee
ARENA_REF = 'main'                                   # or the rules tag, e.g. 'rules-v1.1'
TRAIN_URL = 'https://github.com/partum55/tank-game.git'   # PPO + GRU trainer
os.makedirs(RUN_DIR, exist_ok=True)

In [ ]:
%cd /content
!rm -rf playground tank-game
!git clone -q {ARENA_URL} playground
!git clone -q {TRAIN_URL} tank-game
%cd /content/playground
!git checkout -q {ARENA_REF} && echo "arena:" && git log --oneline | head -1
!cd /content/tank-game && echo "trainer:" && git log --oneline | head -1

## 2. Build the Rust arena (headless: no window libraries needed)

In [ ]:
!curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q
os.environ['PATH'] += ':/root/.cargo/bin'
!pip install -q maturin
!maturin build --release --no-default-features -q -o /content/wheels
!pip install -q --force-reinstall /content/wheels/tank_arena-*.whl

In [ ]:
os.environ['PYTHONPATH'] = '/content/playground/python:/content/tank-game'
%cd /content/tank-game
!python -c "import tank_env; print('rules version', tank_env.RULES_VERSION)"
!python train_ppo.py --self-check

## 3. Train
`--workers 11` leaves one vCPU for the trainer process. A rollout step is ~30k env-steps/s on 12 vCPU; 50M steps ≈ 30–40 min (estimate — read `sps` in the log).
Log columns: `win/loss/draw` = learner vs hall-of-fame + bot over the last 400 episodes, `bot win` = vs the rule-based anchor, `EVAL vs_bot / vs_old` = fixed maps, both sides (1.0 = always wins, 0.5 = even), `*best` = new best export.

In [ ]:
%cd /content/tank-game
!python train_ppo.py --save-dir {RUN_DIR} --workers 11 --envs-per-worker 16 --steps 50e6 --resume

## 4. Take the result to the laptop
`RUN_DIR/export_best/` (and `export_latest/`) contain `agent.py` + `policy.npz` — NumPy only. Download the folder (Drive web UI or the cell below), then on the laptop:

```bash
PYTHONPATH=python python3 python/referee.py export_best/agent.py friend_agent.py --seeds 20   # in the shared repo; the export is self-contained
```
Hall-of-fame snapshots are in `RUN_DIR/hof/` — useful for checking that the newest model really beats the old ones.

In [ ]:
!cd {RUN_DIR} && zip -qr /content/export_best.zip export_best && ls -la /content/export_best.zip
from google.colab import files
files.download('/content/export_best.zip')